In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model=SentenceTransformer("BAAI/bge-m3")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
sentences=[
    "The weather is lovely today.",
    "He drove to the stadium",
    "He battle well today in the cricket match"
]

In [4]:
embeddings=model.encode(sentences)
embeddings

array([[-0.03200633,  0.03941018, -0.05287131, ...,  0.00980747,
        -0.03067565, -0.01396858],
       [ 0.02284115, -0.00185347, -0.03376434, ...,  0.01727378,
         0.02553803,  0.02151975],
       [ 0.01491813,  0.01775536, -0.03060355, ...,  0.00152675,
         0.0377868 , -0.0149782 ]], dtype=float32)

In [5]:
import surprise
import pandas as pd
import numpy as np
from tqdm import tqdm

In [6]:
movies=pd.read_csv("D:\\AshleshaRuchika\\PGCP-AI\\Machine Learning\\archive\\titles.csv",encoding='latin-1')
print(movies.columns)
print(movies.shape)

Index(['id', 'title', 'type', 'description', 'release_year',
       'age_certification', 'runtime', 'genres', 'production_countries',
       'seasons', 'imdb_id', 'imdb_score', 'imdb_votes', 'tmdb_popularity',
       'tmdb_score'],
      dtype='str')
(5850, 15)


In [7]:
movies

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5845,tm1014599,Fine Wine,MOVIE,A beautiful love story that can happen between...,2021,NaN,100,"['romance', 'drama']",['NG'],NaN,tt13857480,6.8,45.0,1.466,NaN
5846,tm898842,C/O Kaadhal,MOVIE,A heart warming film that explores the concept...,2021,NaN,134,['drama'],[],NaN,tt11803618,7.7,348.0,NaN,NaN
5847,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021,NaN,90,['comedy'],['CO'],NaN,tt14585902,3.8,68.0,26.005,6.300
5848,tm1035612,Dad Stop Embarrassing Me - The Afterparty,MOVIE,"Jamie Foxx, David Alan Grier and more from the...",2021,PG-13,37,[],['US'],NaN,NaN,NaN,NaN,1.296,10.000


In [8]:
df_movie=movies[(movies['production_countries'].str.contains('IN')) & (movies['type']=='MOVIE') & (movies['release_year']>=1980)]
df_movie.shape

(570, 15)

In [9]:
df_movie

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
19,tm94651,Dostana,MOVIE,Two close friends decide to enter law enforcem...,1980,NaN,161,"['drama', 'comedy', 'crime', 'romance', 'action']",['IN'],NaN,tt0080653,2.1,25.0,3.980,4.9
56,tm721687,Vaashi,MOVIE,"Ebin Mathew, a budding lawyer ambitiously join...",1983,NaN,123,"['drama', 'thriller']",['IN'],NaN,tt13913068,6.7,388.0,3.790,NaN
65,tm52815,Disco Dancer,MOVIE,"Anil, a street singer, is humiliated and drive...",1982,NaN,106,"['drama', 'romance']",['IN'],NaN,tt0208903,6.4,1476.0,7.879,5.3
66,tm159912,Agneepath,MOVIE,"The movie depicts the life of a young boy, Vij...",1990,NaN,174,"['drama', 'action', 'crime']",['IN'],NaN,tt0098999,7.6,8947.0,4.062,6.5
71,tm157603,Dil,MOVIE,Raja lives a poor lifestyle along with his dad...,1990,NaN,172,"['drama', 'comedy', 'romance']",['IN'],NaN,tt0099429,6.6,5316.0,3.992,6.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5766,tm1004011,Time to Dance,MOVIE,When a ballroom dancerâs shot at a crucial t...,2021,NaN,2,"['romance', 'drama']",['IN'],NaN,tt8622232,2.2,963.0,1.043,4.0
5788,tm987730,Tribhanga,MOVIE,"When her estranged mother falls into a coma, a...",2021,NaN,95,"['drama', 'family']",['IN'],NaN,tt11102314,6.2,2922.0,4.294,7.0
5795,tm988613,Madam Chief Minister,MOVIE,A political-drama where a young woman from a s...,2021,NaN,123,['drama'],['IN'],NaN,tt13773882,4.8,1769.0,2.301,6.7
5821,tm1110241,Kaaval,MOVIE,Thampan and Antony are a long time best friend...,2021,NaN,148,"['thriller', 'action', 'drama']",['IN'],NaN,tt11182984,5.1,1497.0,1.978,5.3


In [10]:
all_desc=list(movies['description'])
out_embs=[]
for r in tqdm(all_desc):
    embd=model.encode(r)
    out_embs.append(embd)
out_embs=np.array(out_embs)
out_embs.shape

 19%|██████████████▊                                                               | 1111/5850 [08:20<35:33,  2.22it/s]


TypeError: 'float' object is not subscriptable

In [ ]:
query="detective investigating a robbery"
qy_emb=model.encode([query])
sims=[]
for e in tqdm(out_embs):
    sims.append(model.similarity(qy_emb,e).numpy()[0])
df_ind=movies[['id','title','description','release_year']].copy()
df_ind.loc[:,'sims']=np.array(sims)
df_ind.sort_values('sims',ascending=Flase)[:5]

In [ ]:
query="detective investigating a robbery"
qy_emb=model.encode([query])
sims=[]
for e in tqdm(out_embs):
    sims.append(model.similarity(qy_emb,e).numpy()[0])
df_ind=movies[['id','title','description','release_year']].copy()
df_ind.loc[:,'sims']=np.array(sims)
df_ind.sort_values('sims',ascending=Flase)[:5]